# Structured Outputs

Many LLM applications do not end with text shown to a human. They feed databases, queues, tickets, dashboards, or other functions. That means the model output must be parseable and validated, not merely plausible.

This notebook explores [structured outputs](https://console.groq.com/docs/structured-outputs) by comparing plain prompting with [JSON](https://www.json.org/json-en.html)-oriented prompting and validating responses with [Pydantic](https://docs.pydantic.dev/latest/).


## Learning Objectives

By the end of this notebook, you should be able to:

- define a small extraction schema,
- call [Groq](https://console.groq.com/docs) or [Mistral](https://docs.mistral.ai/) for structured ticket extraction,
- parse and validate model output with Pydantic,
- distinguish JSON syntax errors from schema errors,
- design a retry prompt that uses validation feedback.


## Structured Data Context

Structured-output prompting is a bridge between natural language and parsers. The parser diagram shows a familiar software-engineering pattern: raw text becomes a representation that code can inspect. Prompt engineering for JSON uses the same idea, except the model produces the intermediate structure.

![Parser transformation diagram](assets/02-parsing-example.png)

Source: [Wikimedia Commons - Parsing Example](https://commons.wikimedia.org/wiki/File:Parsing_Example.png)

The abstract syntax tree image makes the validation goal more concrete. Once information is tree-shaped, a schema or Pydantic model can check field names, data types, allowed values, and missing branches more reliably than it can check prose.

![Abstract syntax tree as a sentence](assets/02-ast-as-sentence.png)

Source: [Wikimedia Commons - AST as a sentence](https://commons.wikimedia.org/wiki/File:AST_as_a_sentence.png)


## Provider And Documentation Notes

[Structured output](https://console.groq.com/docs/structured-outputs) support differs by provider and model. [Groq JSON object mode](https://console.groq.com/docs/structured-outputs) and schema-oriented structured outputs have strict schema support only on selected models. [Mistral JSON mode and custom structured outputs](https://docs.mistral.ai/studio-api/conversations/structured-output) also require explicit JSON instructions in the prompt.

References:

- [Groq structured outputs](https://console.groq.com/docs/structured-outputs)
- [Mistral structured outputs](https://docs.mistral.ai/studio-api/conversations/structured-output)
- [Mistral API specification](https://docs.mistral.ai/api/)


## Shared Provider Helper

Provider selection, OpenAI-compatible client creation, and dry-run chat behavior live in `prompt_helpers.py`. Keeping that plumbing in one place avoids repeating the same setup in every notebook while leaving the prompt, schema, guardrail, and evaluation logic visible here.


In [ ]:
import json
from typing import TypedDict

from prompt_helpers import chat, client_and_model

## Define The Contract First

The schema is the contract between the LLM and the rest of the system. Keep it small at first: fields, valid labels, and types. Add complexity only after you have tests around the current contract.


In [ ]:
from enum import Enum

from pydantic import BaseModel, Field, ValidationError, field_validator


class TicketLabel(str, Enum):
    billing = "BILLING"
    account_access = "ACCOUNT_ACCESS"
    bug_report = "BUG_REPORT"
    feature_request = "FEATURE_REQUEST"
    other = "OTHER"

The enum constrains the labels before we define the full output object. This avoids mixing vocabulary design with field-level validation.



In [ ]:
class TicketExtraction(BaseModel):
    """Structured representation expected from the model."""

    label: TicketLabel
    priority: str = Field(description="low, medium, or high")
    summary: str = Field(min_length=10, max_length=180)
    confidence: float = Field(ge=0, le=1)

    @field_validator("priority")
    @classmethod
    def priority_must_be_known(cls, value: str) -> str:
        normalized = value.lower().strip()
        if normalized not in {"low", "medium", "high"}:
            raise ValueError("priority must be low, medium, or high")
        return normalized

[Pydantic](https://docs.pydantic.dev/latest/) can emit a [JSON Schema](https://json-schema.org/) from the model. We include that schema in the prompt so the provider sees the same contract Python will enforce.



In [ ]:
schema = TicketExtraction.model_json_schema()
print(json.dumps(schema, indent=2))

## Load Example Messages

The dataset is intentionally small so you can inspect every case. In production, this same pattern scales into a larger eval set with expected labels and regression checks.


In [ ]:
class ExtractionExample(TypedDict):
    case_id: str
    text: str
    expected_label: TicketLabel
    expected_priority: str

These cases stay inline because the set is small and used only in this notebook. The expected label and priority fields match the validated `TicketExtraction` contract above; free-form summaries and confidence values are inspected rather than asserted exactly.


In [ ]:
examples: list[ExtractionExample] = [
    {
        "case_id": "extract_001",
        "text": "Customer ACME GmbH reports invoice export failures since Monday. Priority is high.",
        "expected_label": TicketLabel.billing,
        "expected_priority": "high",
    },
    {
        "case_id": "extract_002",
        "text": "User cannot reset their account password because the reset link expired.",
        "expected_label": TicketLabel.account_access,
        "expected_priority": "medium",
    },
]

Add one technical case so extraction is tested beyond billing and account-access messages.


In [ ]:
examples.append(
    {
        "case_id": "extract_003",
        "text": "The prediction API times out during overnight batch scoring.",
        "expected_label": TicketLabel.bug_report,
        "expected_priority": "high",
    }
)
print(json.dumps(examples, indent=2))

## Plain Prompting Is Easy To Read But Hard To Trust

A plain prompt can ask for JSON, but it does not guarantee that the answer will parse or match the schema. We start there because it shows why validation is needed.


In [ ]:
case = examples[0]
plain_prompt = f"""Extract a support ticket record from this message.
Return JSON with label, priority, summary, and confidence.

Message: {case["text"]}"""

plain_response = chat(
    [
        {"role": "system", "content": "You extract compact support ticket records."},
        {"role": "user", "content": plain_prompt},
    ],
    temperature=0.1,
)
print(plain_response)

In [ ]:
def parse_ticket(raw_text: str) -> TicketExtraction:
    """Parse JSON text and validate it against the Pydantic model."""

    data = json.loads(raw_text)
    return TicketExtraction.model_validate(data)


try:
    parsed_plain = parse_ticket(plain_response)
    print(parsed_plain.model_dump())
except (json.JSONDecodeError, ValidationError) as error:
    print(type(error).__name__)
    print(error)

## JSON Mode With Explicit Instructions

For OpenAI-compatible chat completions, `response_format={"type": "json_object"}` requests JSON object mode on providers/models that support it. The prompt still needs to say exactly which JSON object you expect.


In [ ]:
json_prompt = f"""Extract one support ticket record from the message.

Return only a JSON object matching this schema:
{json.dumps(schema, indent=2)}

Rules:
- label must be one of BILLING, ACCOUNT_ACCESS, BUG_REPORT, FEATURE_REQUEST, OTHER.
- priority must be low, medium, or high.
- summary must be one sentence.
- confidence must be between 0 and 1.

Message: {case["text"]}"""

json_response = chat(
    [
        {"role": "system", "content": "You output valid JSON only."},
        {"role": "user", "content": json_prompt},
    ],
    temperature=0.1,
    response_format={"type": "json_object"},
)
print(json_response)

In [ ]:
try:
    parsed_json = parse_ticket(json_response)
    print(parsed_json.model_dump())
except (json.JSONDecodeError, ValidationError) as error:
    print(type(error).__name__)
    print(error)

## Validation Feedback For Retries

A retry should tell the model what failed and preserve the original input. Do not silently repair data unless the repair rule is deterministic and documented.


In [ ]:
def build_retry_prompt(
    original_message: str, bad_response: str, error: Exception
) -> str:
    """Create a focused retry prompt from validator feedback."""

    return f"""The previous response did not pass validation.

Original message:
{original_message}

Invalid response:
{bad_response}

Validation error:
{error}

Return only corrected JSON matching the schema:
{json.dumps(schema, indent=2)}"""


bad_response = '{"label": "LOGIN", "priority": "urgent", "summary": "Locked out", "confidence": 1.2}'
try:
    parse_ticket(bad_response)
except Exception as error:
    retry_prompt = build_retry_prompt(case["text"], bad_response, error)
    print(retry_prompt[:900])

## Batch Extraction Pattern

For a real workflow, run extraction over multiple cases, validate each result, and keep the raw response next to the parsed object. That makes failures auditable.


In [ ]:
def extract_ticket(message: str) -> dict[str, object]:
    """Call the model once and return both raw and validated results."""

    prompt = f"""Return a JSON support ticket matching this schema:
{json.dumps(schema, indent=2)}

Message: {message}"""
    raw = chat(
        [
            {"role": "system", "content": "You output valid JSON only."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1,
        response_format={"type": "json_object"},
    )
    try:
        parsed = parse_ticket(raw)
        return {"ok": True, "raw": raw, "parsed": parsed.model_dump()}
    except (json.JSONDecodeError, ValidationError) as error:
        return {"ok": False, "raw": raw, "error": str(error)}

Run every inline example and compare the validated label and priority to the expected contract fields. Summaries and confidence values remain available for inspection rather than exact-match scoring.


In [ ]:
def check_example(item: ExtractionExample) -> dict[str, object]:
    """Evaluate validated categorical fields against one expected result."""

    result = extract_ticket(item["text"])
    if not result["ok"]:
        return {"case_id": item["case_id"], "matches_expected": False, "result": result}
    parsed = TicketExtraction.model_validate(result["parsed"])
    matches_expected = (
        parsed.label == item["expected_label"]
        and parsed.priority == item["expected_priority"]
    )
    return {
        "case_id": item["case_id"],
        "matches_expected": matches_expected,
        "parsed": parsed.model_dump(),
    }


checked_results = [check_example(item) for item in examples]
print(json.dumps(checked_results, indent=2, default=str))

## Exercise

Add a new field named `requires_human_review` to the schema. It should be a boolean. Update the prompt and validator, then run the extractor on three messages.

A good solution explains when the model should set the value to `true`, for example high-risk billing disputes, account lockouts, or unclear messages.


## Correction

The additional boolean belongs in the validated contract, not only in the prompt text. This version requires review for requests that carry operational risk or cannot be routed confidently.


In [ ]:
class ReviewedTicketExtraction(TicketExtraction):
    """Extraction record with an explicit human-review decision."""

    requires_human_review: bool = Field(
        description="True for high-risk, blocked, or ambiguous support cases."
    )


review_schema = ReviewedTicketExtraction.model_json_schema()
print(json.dumps(review_schema["properties"]["requires_human_review"], indent=2))

The extractor below requests the new field. Its local fallback is deterministic so the correction remains runnable without an API key; with a configured provider, the same validator checks the live JSON response.


In [ ]:
def dry_review_payload(message: str) -> dict[str, object]:
    """Build a deterministic example record when no provider key is set."""

    lowered = message.lower()
    risky = any(word in lowered for word in ["locked", "dispute", "unclear"])
    if any(word in lowered for word in ["charge", "invoice", "refund"]):
        label = "BILLING"
    elif any(word in lowered for word in ["account", "mfa", "login"]):
        label = "ACCOUNT_ACCESS"
    elif any(word in lowered for word in ["add", "setting"]):
        label = "FEATURE_REQUEST"
    else:
        label = "OTHER"
    return {
        "label": label,
        "priority": "medium",
        "summary": message,
        "confidence": 0.72,
        "requires_human_review": risky,
    }

Both live and dry responses pass through the same Pydantic validation step, so the schema remains the enforcement point.


In [ ]:
def extract_review_ticket(message: str) -> ReviewedTicketExtraction:
    """Extract and validate a ticket that includes the review decision."""

    prompt = f"""Return JSON matching this schema:
{json.dumps(review_schema, indent=2)}

Set requires_human_review to true for account lockouts, disputed charges, or unclear routing.
Message: {message}"""
    client, _, _, dry_reason = client_and_model()
    if client is None:
        print(dry_reason)
        raw = json.dumps(dry_review_payload(message))
    else:
        raw = chat(
            [{"role": "user", "content": prompt}],
            temperature=0.1,
            response_format={"type": "json_object"},
        )
    return ReviewedTicketExtraction.model_validate_json(raw)

In [ ]:
review_messages = [
    "The account is locked after an MFA reset request.",
    "The customer disputes a renewal charge on the invoice.",
    "Please add a dark mode setting to the dashboard.",
]
for message in review_messages:
    print(extract_review_ticket(message).model_dump())

## Further Reading

- [JSON Schema documentation](https://json-schema.org/learn/getting-started-step-by-step)
- [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/)
- [Groq structured outputs](https://console.groq.com/docs/structured-outputs)
- [Mistral structured outputs](https://docs.mistral.ai/studio-api/conversations/structured-output)
- [Outlines structured generation](https://dottxt-ai.github.io/outlines/latest/)

